# PE-U4: pandas vs PySpark — escalabilidad y Ley de Amdahl (dominio ACC)

Implementación y comparación de **5 transformaciones (T1–T5)** sobre un dataset
sintético de **600 000 tickets de soporte técnico ISP** (PFC ACC) en **pandas**
y **PySpark 3.5.3**, con protocolo de medición riguroso (1 calentamiento + 5
repeticiones, mediana, `time.perf_counter`), escalado de T3 con **N = 1, 2 y 4
executors** y análisis de **Amdahl**.

Plataforma: **Google Colab** (requisito del checklist). El dataset y los
módulos de `src/` deben estar accesibles en el directorio de trabajo.


## PFC de referencia

- **Código:** ACC
- **Título del sistema:** Sistema de Gestión de Soporte Técnico ISP
- **Integrantes / matrículas / grupo:** *pendiente de completar* (portada e
  informe LaTeX diferidos).


## 0. Instalación del entorno (Colab)

PySpark 3.5.3 requiere Java (se instala JDK 11). Las versiones exactas quedan
fijadas por `requirements.txt` del proyecto.


## 1. Ubicación del proyecto y del dataset

La celda siguiente **localiza la carpeta del proyecto automáticamente** (la que
contiene `src/` y `data/`) entre las ubicaciones habituales de Colab:

* `/content/drive/MyDrive/GA-SUM-05` (si subiste la carpeta `Soporte-Tecnico-ACC`
  dentro de la carpeta `GA-SUM-05` de tu Google Drive; Colab la monta al abrir
  el notebook desde Drive o con el icono de Drive).
* `/content/drive/MyDrive` (si subiste el proyecto directamente a la raíz de
  Drive).
* `/content` (si arrastraste la carpeta o la subiste con el panel de archivos).

No es necesario escribir rutas ni montar Drive a mano.


## 2. Sesión Spark (criterio 1.5, checklist 15-16)

- `spark.executor.instances = 4` se configura (requisito del checklist). En
  `local[N]` Spark crea **1 executor con N cores**: la paralelización efectiva
  es `local[4]` y el escalado se simula con `local[1]`, `local[2]`, `local[4]`.
- `spark.sql.autoBroadcastJoinThreshold = -1` desactiva el broadcast join para
  que T3 (join con la tabla pequeña `agentes`) haga **shuffle real**
  (sort-merge) y el escalado/Amdahl sean observables.


## 3. Lectura del dataset (criterio 1.2)

`pandas.describe()` + `head()` y `Spark.printSchema()` con **esquema fijado
explícitamente** (ninguna columna se infiere distinta). Semántica de nulos
idéntica en ambos motores: `""` -> `NaN` (pandas) / `NULL` (Spark).


## 4. Esquema y tamaño: tabla de comparación (criterio 1.7)


## 5. Transformaciones T1–T5 (criterio 1.3)

Cada transformación es **independiente** (parte del crudo) y fuerza
**materialización** (`.write.csv` / `to_csv`): nunca se mide un DAG perezoso.

| Id | Operación | Descripción |
|---|---|---|
| T1 | Filtrado | Tickets RESUELTO/CERRADO por WEB/APP con agente, resolución y zona no nulos |
| T2 | Agrupación | 6 agregaciones por `(categoria, zona)` |
| T3 | Join | Inner `tickets` ⨝ `agentes` por `agente_id` (sort-merge, sin broadcast) |
| T4 | Derivadas | `anio/mes/dia`, tiempo de respuesta (h), SLA cumplido, prioridad crítica, asunto en mayúsculas |
| T5 | Top-N | Los 10 asuntos con más tickets |

**Salidas:** `data/pandas/*.csv` y `data/spark/<transform>/part-*.csv`
(no versionadas en el repositorio).


## 6. Equivalencia numérica pandas ↔ Spark (criterio 1.4)

Compara **cardinalidad**, **columnas**, **sumas y medias** (tolerancia relativa
1e-6) y **claves de grupos** (T2/T5). La tabla comparativa se muestra a
continuación y se transcribe al informe (no se persiste como fichero).


## 7. Protocolo de medición (criterios 1.3, 1.6, 2.1)

- 1 ejecución de **calentamiento** descartada por (transformación, motor).
- **5 repeticiones** cronometradas con `time.perf_counter()`.
- Estadístico reportado: **mediana**.
- Se mide el ciclo completo **lectura + limpieza + transformación +
  persistencia**; el arranque de la SparkSession **no** se cronometra.
- T3 en Spark se mide con **N = 1, 2 y 4**.

> La ejecución completa tarda **~30 min** en Colab. Si solo se quiere repetir
> una parte: `--solo pandas` o `--solo spark`. Los tiempos finales deben
> generarse **íntegros en Colab** (mismo entorno) para el informe.


## 8. Análisis de Amdahl (criterio 2.2)

A partir de T3 (N=1,2,4): `S_vs_pandas`, `S_interno`, fracción serial
`f = (1/S_int - 1/N)/(1 - 1/N)`, `p = 1-f`, `S_max = 1/f`, eficiencia
`E(N) = S_int/N` y Gustafson-Barsis. La tabla se muestra a continuación y se
transcribe al informe (no se persiste como fichero).


## 9. Umbral de rentabilidad Spark vs pandas (criterio 2.4)

Se mide T2 (agrupación) con muestreo determinista (primeras `tam` filas del
crudo) a 10k, 50k, 100k, 250k, 500k y 600k filas: 1 calentamiento + 3
repeticiones, mediana. La tabla se muestra a continuación y se transcribe al
informe (no se persiste como fichero).


## 10. Versiones exactas del entorno (reproducibilidad, criterio 1.4)

Se imprimen a continuación y se transcriben al informe (no se persiste como
fichero).


## 11. Parquet: CSV vs Parquet (criterio 5.1/5.2)

Se materializa T1 en **Parquet** (pandas y Spark), se vuelve a leer y se
compara tamaño en disco y coste de lectura frente al CSV.


## 12. Spark UI: evidencia (criterio 1.5, checklist 10)

La Spark UI (puerto **4040**) queda habilitada (`spark.ui.enabled = true`).
Durante una corrida de T3 la sesión está viva varios minutos: es el momento de
capturar la evidencia (DAG + stages del sort-merge).

**Método recomendado (sin cuentas ni instalaciones):** en el menú de Colab
*Herramientas → Puertos locales (Tools → Local port forwarding)* añade el
puerto **4040** y pulsa el enlace que genera. Se abre la Spark UI en el
navegador.

**Método alternativo (ngrok, requiere cuenta gratuita):**

```python
# !pip install -q pyngrok
# from pyngrok import ngrok
# ngrok.kill()
# tunel = ngrok.connect(4040)
# print("Spark UI:", tunel.public_url)
```

Pasos para la evidencia:
1. Durante la ejecución de la **celda 34** (protocolo completo, ~30 min) o de la
   **celda 32** (T3 con N=1,2,4) abre el puerto 4040 con Local port forwarding.
2. Entra en la pestaña **SQL / Stages** (DAG + stages del sort-merge de T3) y
   haz la captura de pantalla.
3. Guarda la imagen como **`evidencia/spark_ui_t3.png`** y tráela al proyecto.


## 13. Figuras (criterio 2.10)

```python
!python src/graficas.py
```

Genera `resultados/figuras/fig0…fig3_*.png` a **300 DPI** a partir de
`tiempos_resumen.csv` (los indicadores de Amdahl se calculan dentro de
`graficas.py`; no se persiste ningún CSV intermedio).


## 14. Conclusión y evidencia

- `resultados/`: `tiempos_crudos.csv`, `tiempos_resumen.csv` y `figuras/`.
- Las tablas derivadas (equivalencia, Amdahl, umbral y versiones) se
  transcriben al **informe LaTeX** (criterios 1.4, 2.2, 2.4).
- `evidencia/spark_ui_t3.png`: captura de la Spark UI.
- Descarga el notebook ejecutado con `Archivo → Descargar → Descargar .ipynb`
  y guárdalo como `notebooks/PE_U4_pipeline_spark.ipynb` (el `.html` se genera
  en local a partir de él).

Declaración de uso de IA generativa: ver el informe LaTeX (sección final).


## 15. Verificación de evidencia y empaquetado (Colab)

La última celda comprueba qué artefactos del checklist existen en disco,
**rellena automáticamente `tablas_items_4_12_13_20_23.txt`** con las tablas
reales (items 4/12/13/20/23) y empaqueta `resultados/` + `evidencia/` + el
`.txt` en `resultados_pe_u4.zip` para descargarlo. Reutiliza las variables de
las celdas 33 (equivalencia), 36 (Amdahl), 37 (umbral) y 38 (versiones): **no
repite las mediciones**. Descarga el zip y `tablas_items_4_12_13_20_23.txt`
para llevarlos al repositorio local.


In [3]:
import glob
import os
import subprocess
from pathlib import Path

def _sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(f"FALLO en: {cmd}\n--- stderr (final) ---\n{r.stderr[-2000:]}")

def _buscar_requirements():
    conocidas = [
        Path(os.getcwd()) / "requirements.txt",
        Path("/content/Soporte-Tecnico-ACC/requirements.txt"),
        Path("/content/drive/MyDrive/Soporte-Tecnico-ACC/requirements.txt"),
        Path("/content/drive/MyDrive/GA-SUM-05/Soporte-Tecnico-ACC/requirements.txt"),
    ]
    for p in conocidas:
        if p.exists():
            return p.resolve()
    def _caminar(base, prof=0):
        if prof > 3 or not base.is_dir():
            return None
        cand = base / "requirements.txt"
        if cand.exists():
            return cand.resolve()
        for sub in base.iterdir():
            if sub.is_dir():
                r = _caminar(sub, prof + 1)
                if r is not None:
                    return r
        return None
    for base in (Path("/content"), Path("/content/drive/MyDrive")):
        if base.is_dir():
            r = _caminar(base)
            if r is not None:
                return r
    return None

_req = _buscar_requirements()
if _req is None:
    raise SystemExit(
        "requirements.txt no encontrado. Sube la carpeta Soporte-Tecnico-ACC "
        "dentro de GA-SUM-05 en Drive o al panel de archivos de Colab."
    )

# 1) Java (PySpark 3.5 requiere JDK 8/11/17). Se instala si no hay java.
r = subprocess.run("java -version", shell=True, capture_output=True, text=True)
if r.returncode != 0:
    _sh("apt-get update -qq && apt-get install -y -qq openjdk-11-jdk-headless")
else:
    _sh("apt-get install -y -qq openjdk-11-jdk-headless || true")

# 2) PySpark limpio: quita cualquier pyspark previo (Colab trae uno) y lo
#    reinstala completo (con el jar de spark-sql) para evitar instalaciones
#    parciales que rompen la SparkSession.
_sh("pip uninstall -y -q pyspark py4j 2>/dev/null || true")
_sh('pip install -q --no-cache-dir "pyspark==3.5.3" "py4j==0.10.9.7"')

# 3) Librerías compiladas con --force-reinstall: limpia binarios mezclados
#    (numpy 1.26 + 2.x) que rompen `import pandas` ("numpy.dtype size changed").
_compiladas = ["numpy==2.0.2", "pandas==2.2.3", "matplotlib==3.9.2"]
for _pkg in _compiladas:
    _sh(f"pip install -q --no-cache-dir --force-reinstall {_pkg!r}")

# 4) Resto de requirements.txt (sin pyspark/py4j ni las compiladas de arriba).
_excluidos = ("pyspark", "py4j", "numpy", "pandas", "matplotlib")
deps = []
for ln in Path(_req).read_text(encoding="utf-8").splitlines():
    ln = ln.strip()
    if not ln or ln.startswith("#"):
        continue
    if any(k in ln.lower() for k in _excluidos):
        continue
    deps.append(ln)
if deps:
    _sh("pip install -q --no-cache-dir " + " ".join(f'"{d}"' for d in deps))
print("Instalacion completada (PySpark 3.5.3 + stack numpy 2).")


Instalacion completada (PySpark 3.5.3 + stack numpy 2).


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import glob
import os
import sys
from pathlib import Path

jdks = sorted(set(
    glob.glob("/usr/lib/jvm/java-11-*")
    + glob.glob("/usr/lib/jvm/java-17-*")
    + glob.glob("/usr/lib/jvm/jdk-11*")
    + glob.glob("/usr/lib/jvm/jdk-17*")
))
if jdks:
    os.environ["JAVA_HOME"] = jdks[0]
print("JAVA_HOME =", os.environ.get("JAVA_HOME"))

try:
    import numpy as np
    print("numpy  ", np.__version__)
except ValueError as _e:
    print("Error de numpy:", _e)
    print("Binarios mezclados. Reinstala limpio y reinicia la sesion:")
    print('  !pip install -q --no-cache-dir --force-reinstall numpy==2.0.2 pandas==2.2.3 matplotlib==3.9.2')
    print("Luego: Entorno de ejecucion -> Reiniciar sesion y ejecutar todo.")
    raise SystemExit("numpy roto (binarios mezclados).")

import pyspark
print("pyspark", pyspark.__version__)

jars_sql = glob.glob(str(Path(pyspark.__file__).resolve().parent / "jars" / "spark-sql_*.jar"))
if not jars_sql:
    print("FALTA jars/spark-sql_*.jar: la instalacion de pyspark quedo incompleta.")
    print("Reinstala con:")
    print('  !pip uninstall -y -q pyspark py4j && pip install -q pyspark==3.5.3 py4j==0.10.9.7')
    print("y luego: Entorno de ejecucion -> Reiniciar sesion y ejecutar todo.")
    raise SystemExit("pyspark incompleto: no existe jars/spark-sql_*.jar.")
print("jars/spark-sql:", os.path.basename(jars_sql[0]))

try:
    import pandas as pd
    print("pandas ", pd.__version__)
except ValueError as _e:
    print("Error de pandas:", _e)
    print("Binarios mezclados. Reinstala limpio y reinicia la sesion:")
    print('  !pip install -q --no-cache-dir --force-reinstall numpy==2.0.2 pandas==2.2.3 matplotlib==3.9.2')
    print("Luego: Entorno de ejecucion -> Reiniciar sesion y ejecutar todo.")
    raise SystemExit("pandas roto (binarios mezclados).")


JAVA_HOME = /usr/lib/jvm/java-11-openjdk-amd64
numpy   2.0.2
pyspark 3.5.3
jars/spark-sql: spark-sql_2.12-3.5.3.jar
pandas  2.2.3


In [6]:
import os
import sys
from pathlib import Path

def _es_proyecto(p):
    return (p / "src" / "transformaciones_pandas.py").exists() and (p / "data" / "raw").is_dir()

def _buscar():
    conocidas = [
        Path(os.getcwd()),
        Path("/content/Soporte-Tecnico-ACC"),
        Path("/content/drive/MyDrive/Soporte-Tecnico-ACC"),
        Path("/content/drive/MyDrive/GA-SUM-05/Soporte-Tecnico-ACC"),
    ]
    for p in conocidas:
        if _es_proyecto(p):
            return p
    def _caminar(base, prof=0):
        if prof > 3 or not base.is_dir():
            return None
        if _es_proyecto(base):
            return base
        for sub in base.iterdir():
            if sub.is_dir():
                r = _caminar(sub, prof + 1)
                if r is not None:
                    return r
        return None
    for base in (Path("/content"), Path("/content/drive/MyDrive")):
        if base.is_dir():
            r = _caminar(base)
            if r is not None:
                return r
    return None

root = _buscar()

if root is None:
    raise SystemExit(
        "No se encontro el proyecto (src/transformaciones_pandas.py + data/raw). "
        "Sube la carpeta Soporte-Tecnico-ACC dentro de GA-SUM-05 en Drive o a /content."
    )

os.chdir(root)
sys.path.insert(0, str(root / "src"))
print("ROOT =", root)


ROOT = /content/drive/My Drive/GA-SUM-05/Soporte-Tecnico-ACC


In [7]:
from pathlib import Path

tickets = Path("data/raw/tickets.csv")
agentes = Path("data/raw/agentes.csv")
assert tickets.exists(), "Falta data/raw/tickets.csv (sube el dataset a Colab)"
assert agentes.exists(), "Falta data/raw/agentes.csv"
print(f"tickets.csv: {tickets.stat().st_size / 1e6:.1f} MB")
print(f"agentes.csv: {agentes.stat().st_size / 1e6:.2f} MB")


tickets.csv: 75.8 MB
agentes.csv: 0.02 MB


In [8]:
from transformaciones_spark import crear_spark

spark = crear_spark(4, nombre="pe-u4-notebook")
print("Motor Spark:", spark.version)
print("master     : local[4]")


Motor Spark: 3.5.3
master     : local[4]


In [9]:
conf = spark.sparkContext.getConf()

# (a) configuracion COMPLETA de la sesion (criterio 1.5): getConf().getAll()
print("=== getConf().getAll() ===")
for k, v in sorted(conf.getAll()):
    print(f"{k:50s} = {v}")

print()
print("=== claves relevantes del checklist ===")
claves = [
    "spark.master",
    "spark.app.name",
    "spark.executor.instances",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.shuffle.partitions",
    "spark.sql.session.timeZone",
    "spark.ui.enabled",
    "spark.driver.host",
]
for k in claves:
    print(f"{k:45s} = {conf.get(k)}")


=== getConf().getAll() ===
spark.app.id                                       = local-1786197771628
spark.app.name                                     = pe-u4-notebook (N=4)
spark.app.startTime                                = 1786197769856
spark.app.submitTime                               = 1786197769236
spark.driver.bindAddress                           = 127.0.0.1
spark.driver.extraJavaOptions                      = -Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.ba

In [10]:
import pandas as pd

from transformaciones_pandas import leer_crudo_pandas

raw_pd = leer_crudo_pandas()
print("pandas shape:", raw_pd.shape)
raw_pd.head(5)


pandas shape: (600000, 15)


,ticket_id,cliente_id,agente_id,asunto,categoria,estado,prioridad,canal,zona,sla_horas,fecha_creacion,fecha_resolucion,hora_creacion,dia_semana_creacion,mes_creacion
0,TK-00000001,8558,AG-208,No funciona el WiFi,EQUIPO,CERRADO,MEDIA,TELEFONO,NORTE,48,2025-06-09 15:31:51,2025-06-11 06:52:51,15,1,6
1,TK-00000002,177756,AG-173,No hay conexion a Internet,FACTURACION,RESUELTO,MEDIA,WEB,SUR,48,2026-04-10 23:19:32,2026-04-12 13:00:32,23,5,4
2,TK-00000003,97224,AG-033,Conexion intermitente,EQUIPO,CERRADO,ALTA,WEB,NORTE,24,2026-03-21 01:58:50,2026-03-21 04:06:50,1,6,3
3,TK-00000004,158095,AG-068,Factura incorrecta,FACTURACION,RESUELTO,ALTA,APP,NORTE,24,2026-04-08 12:30:00,2026-04-10 04:12:00,12,3,4
4,TK-00000005,87735,NaN,Reclamacion por cobro,INSTALACION,CREADO,ALTA,EMAIL,NORTE,24,2026-05-14 03:25:31,NaT,3,4,5


In [11]:
raw_pd[["cliente_id", "sla_horas", "hora_creacion", "dia_semana_creacion", "mes_creacion"]].describe()


,cliente_id,sla_horas,hora_creacion,dia_semana_creacion,mes_creacion
count,600000.000000,600000.000000,600000.000000,600000.000000,600000.000000
mean,99894.733002,43.580407,11.501597,4.004083,6.532003
std,57747.662444,21.494915,6.920699,1.999126,3.447578
min,1.000000,4.000000,0.000000,1.000000,1.000000
25%,49789.000000,24.000000,6.000000,2.000000,4.000000
50%,99958.000000,48.000000,12.000000,4.000000,7.000000
75%,149800.000000,48.000000,18.000000,6.000000,10.000000
max,200000.000000,72.000000,23.000000,7.000000,12.000000


In [12]:
from transformaciones_spark import leer_crudo_spark

raw_sp = leer_crudo_spark(spark)
print("Spark count:", raw_sp.count())
raw_sp.printSchema()


Spark count: 600000
root
 |-- ticket_id: string (nullable = true)
 |-- cliente_id: double (nullable = true)
 |-- agente_id: string (nullable = true)
 |-- asunto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- prioridad: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- sla_horas: double (nullable = true)
 |-- fecha_creacion: timestamp (nullable = true)
 |-- fecha_resolucion: timestamp (nullable = true)
 |-- hora_creacion: integer (nullable = true)
 |-- dia_semana_creacion: integer (nullable = true)
 |-- mes_creacion: integer (nullable = true)



In [13]:
raw_sp.select("ticket_id", "estado", "prioridad", "canal", "zona").show(5, truncate=False)


+-----------+--------+---------+--------+-----+
|ticket_id  |estado  |prioridad|canal   |zona |
+-----------+--------+---------+--------+-----+
|TK-00000001|CERRADO |MEDIA    |TELEFONO|NORTE|
|TK-00000002|RESUELTO|MEDIA    |WEB     |SUR  |
|TK-00000003|CERRADO |ALTA     |WEB     |NORTE|
|TK-00000004|RESUELTO|ALTA     |APP     |NORTE|
|TK-00000005|CREADO  |ALTA     |EMAIL   |NORTE|
+-----------+--------+---------+--------+-----+
only showing top 5 rows



In [14]:
from transformaciones_pandas import COLUMNAS

tipos_spark = dict(raw_sp.dtypes)
tabla_esquema = pd.DataFrame({
    "columna": COLUMNAS,
    "tipo_pandas": [str(raw_pd[c].dtype) for c in COLUMNAS],
    "tipo_spark": [tipos_spark[c] for c in COLUMNAS],
})
tabla_esquema


,columna,tipo_pandas,tipo_spark
0,ticket_id,object,string
1,cliente_id,int64,double
2,agente_id,object,string
3,asunto,object,string
4,categoria,object,string
5,estado,object,string
6,prioridad,object,string
7,canal,object,string
8,zona,object,string
9,sla_horas,int64,double


In [15]:
print(f"tickets.csv en disco: {tickets.stat().st_size / 1e6:.1f} MB | 600 000 filas x 15 columnas")
print(f"agentes.csv en disco: {agentes.stat().st_size / 1e6:.2f} MB | 300 filas x 7 columnas")


tickets.csv en disco: 75.8 MB | 600 000 filas x 15 columnas
agentes.csv en disco: 0.02 MB | 300 filas x 7 columnas


In [16]:
from transformaciones_pandas import PANDAS_OUT, PANDAS_PATHS, TRANSFORMACIONES

PANDAS_OUT.mkdir(parents=True, exist_ok=True)
for t, f in TRANSFORMACIONES.items():
    r = f(raw_pd)
    r.to_csv(PANDAS_PATHS[t], index=False, encoding="utf-8")
    print(f"{t}: {len(r):,} filas x {r.shape[1]} col -> {PANDAS_PATHS[t].name}")


T1: 180,003 filas x 9 col -> t1_filtrado.csv
T2: 30 filas x 8 col -> t2_agrupacion.csv
T3: 180,003 filas x 11 col -> t3_join.csv
T4: 600,000 filas x 9 col -> t4_derivada.csv
T5: 10 filas x 2 col -> t5_topn.csv


In [17]:
from transformaciones_spark import SPARK_OUT, SPARK_PATHS, TRANSFORMACIONES

SPARK_OUT.mkdir(parents=True, exist_ok=True)
for t in ["T1", "T2", "T4", "T5"]:
    r = TRANSFORMACIONES[t](raw_sp)
    r.write.mode("overwrite").csv(str(SPARK_PATHS[t]), header=True, encoding="utf-8")
    print(f"{t}: {r.count():,} filas materializadas -> {SPARK_PATHS[t].name}")


T1: 180,003 filas materializadas -> t1_filtrado
T2: 30 filas materializadas -> t2_agrupacion
T4: 600,000 filas materializadas -> t4_derivada
T5: 10 filas materializadas -> t5_topn


In [18]:
# Cierra la sesion local[4] de la celda 22 antes de escalar: SparkSession
# .getOrCreate() devuelve la sesion ACTIVA e ignora el master del builder.
# Sin este stop(), el bucle ejecutaria T3 tres veces en local[4] y el escalado
# N=1,2,4 (checklist 15) no se mediria realmente en este notebook.
spark.stop()

for n in [1, 2, 4]:
    sp = crear_spark(n, nombre="t3-notebook")
    crudo = leer_crudo_spark(sp)
    r = TRANSFORMACIONES["T3"](crudo)
    r.write.mode("overwrite").csv(str(SPARK_PATHS["T3"]), header=True, encoding="utf-8")
    print(f"T3 (N={n}): {r.count():,} filas materializadas")
    sp.stop()


T3 (N=1): 180,003 filas materializadas
T3 (N=2): 180,003 filas materializadas
T3 (N=4): 180,003 filas materializadas


In [19]:
import medicion as m

df_equivalencia = m.modo_equivalencia()
df_equivalencia


T1: filas=180003/180003 equivalentes=True
T2: filas=30/30 equivalentes=True
T3: filas=180003/180003 equivalentes=True
T4: filas=600000/600000 equivalentes=True
T5: filas=10/10 equivalentes=True
Tabla comparativa (criterio 1.4):
transform  filas_pandas  filas_spark  columnas_ok  cardinalidad_ok  sumas_ok  medias_ok  claves_ok  equivalente
       T1        180003       180003         True             True      True       True       True         True
       T2            30           30         True             True      True       True       True         True
       T3        180003       180003         True             True      True       True       True         True
       T4        600000       600000         True             True      True       True       True         True
       T5            10           10         True             True      True       True       True         True


,transform,filas_pandas,filas_spark,columnas_ok,cardinalidad_ok,sumas_ok,medias_ok,claves_ok,equivalente
0,T1,180003,180003,True,True,True,True,True,True
1,T2,30,30,True,True,True,True,True,True
2,T3,180003,180003,True,True,True,True,True,True
3,T4,600000,600000,True,True,True,True,True,True
4,T5,10,10,True,True,True,True,True,True


In [20]:
!python src/medicion.py --modo medicion


--- pandas ---
T1: mediana=7.567s (n=5)
T2: mediana=6.311s (n=5)
T3: mediana=8.313s (n=5)
T4: mediana=11.229s (n=5)
T5: mediana=6.485s (n=5)
-> provisional: /content/drive/My Drive/GA-SUM-05/Soporte-Tecnico-ACC/resultados/tiempos_crudos.csv y /content/drive/My Drive/GA-SUM-05/Soporte-Tecnico-ACC/resultados/tiempos_resumen.csv
--- spark ---
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/08 14:09:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
T1 (N=4): mediana=5.437s
T2 (N=4): mediana=4.360s
T4 (N=4): mediana=6.244s
T5 (N=4): mediana=2.205s
-> provisional: /content/drive/My Drive/GA-SUM-05/Soporte-Tecnico-ACC/resultados/tiempos_crudos.csv y /content/drive/My Drive/GA-SUM-05/Soporte-Tecnico-ACC/resultados/tiempos_resumen.csv
T3 (N=1): mediana=6.287s
T3 (N=2): mediana=6.167s
T3 (N=4): mediana=7.410s
-> provisional: /conten

In [21]:
import pandas as pd

pd.read_csv("resultados/tiempos_resumen.csv").to_string(index=False)


'transform engine  n_cores  median_s    mean_s    std_s  speedup\n       T1 pandas        0  7.566817  8.386852 1.381461      NaN\n       T2 pandas        0  6.311093  6.707256 1.108996      NaN\n       T3 pandas        0  8.313494  8.876503 0.889813      NaN\n       T4 pandas        0 11.228543 11.286367 1.223910      NaN\n       T5 pandas        0  6.485240  6.424569 0.861551      NaN\n       T1  spark        4  5.437160  5.839110 1.019556 1.391686\n       T2  spark        4  4.360306  4.294998 1.163171 1.447397\n       T4  spark        4  6.243921  7.302071 1.525488 1.798316\n       T5  spark        4  2.205043  2.630063 0.771868 2.941095\n       T3  spark        1  6.287495  6.899384 1.576068 1.322227\n       T3  spark        2  6.167184  6.578907 1.228063 1.348021\n       T3  spark        4  7.409939  7.168126 0.993216 1.121938'

In [22]:
import medicion as m

df_amdahl = m.modo_amdahl()
df_amdahl


T3 pandas (base): 8.313 s | T3 spark N=1: 6.287 s
N=1: S_vs_pandas=1.322  S_interno=1.000  E=1.000  f=---
N=2: S_vs_pandas=1.348  S_interno=1.020  E=0.510  f=0.962
N=4: S_vs_pandas=1.122  S_interno=0.849  E=0.212  f=1.238
p (fracción paralela, S_int) = -0.0999 | S_max = 0.9092
Tabla de Amdahl (criterio 2.2):
 N  t_pandas_s  t_spark_s  speedup_vs_pandas  speedup_interno  f_serial_estimada  eficiencia_E  f_serial_media  p_paralela  S_max_amdahl  S_gustafson
 1    8.313494   6.287495           1.322227         1.000000                NaN      1.000000        1.099878   -0.099878      0.909191     1.000000
 2    8.313494   6.167184           1.348021         1.019508           0.961730      0.509754        1.099878   -0.099878      0.909191     0.900122
 4    8.313494   7.409939           1.121938         0.848522           1.238027      0.212130        1.099878   -0.099878      0.909191     0.700365


,N,t_pandas_s,t_spark_s,speedup_vs_pandas,speedup_interno,f_serial_estimada,eficiencia_E,f_serial_media,p_paralela,S_max_amdahl,S_gustafson
0,1,8.313494,6.287495,1.322227,1.000000,NaN,1.000000,1.099878,-0.099878,0.909191,1.000000
1,2,8.313494,6.167184,1.348021,1.019508,0.961730,0.509754,1.099878,-0.099878,0.909191,0.900122
2,4,8.313494,7.409939,1.121938,0.848522,1.238027,0.212130,1.099878,-0.099878,0.909191,0.700365


In [23]:
import medicion as m

df_umbral = m.modo_umbral()
df_umbral


tam= 10,000: pandas=7.419s  spark=1.233s  SPARK<pandas
tam= 50,000: pandas=7.538s  spark=2.130s  SPARK<pandas
tam=100,000: pandas=5.589s  spark=3.447s  SPARK<pandas
tam=250,000: pandas=5.897s  spark=3.874s  SPARK<pandas
tam=500,000: pandas=6.131s  spark=4.362s  SPARK<pandas
tam=600,000: pandas=6.640s  spark=4.548s  SPARK<pandas
Punto de cruce: Spark supera a pandas a partir de 10,000 filas.
Tabla del umbral de rentabilidad (criterio 2.4):
 tamano engine  median_s   mean_s    std_s  crossover_size
  10000 pandas  7.418576 6.896784 1.039248           10000
  10000  spark  1.232536 1.274639 0.206331           10000
  50000 pandas  7.537650 7.233865 1.462658           10000
  50000  spark  2.129506 2.161677 0.065079           10000
 100000 pandas  5.588745 6.714792 1.975059           10000
 100000  spark  3.447198 4.156956 1.298137           10000
 250000 pandas  5.896558 6.490131 1.273897           10000
 250000  spark  3.874307 5.074214 2.106805           10000
 500000 pandas  6.130606 6

,tamano,engine,median_s,mean_s,std_s,crossover_size
0,10000,pandas,7.418576,6.896784,1.039248,10000
1,10000,spark,1.232536,1.274639,0.206331,10000
2,50000,pandas,7.537650,7.233865,1.462658,10000
3,50000,spark,2.129506,2.161677,0.065079,10000
4,100000,pandas,5.588745,6.714792,1.975059,10000
5,100000,spark,3.447198,4.156956,1.298137,10000
6,250000,pandas,5.896558,6.490131,1.273897,10000
7,250000,spark,3.874307,5.074214,2.106805,10000
8,500000,pandas,6.130606,6.835593,1.350677,10000
9,500000,spark,4.362101,4.595094,0.725865,10000


In [24]:
import medicion as m

versiones = m.modo_versiones()
versiones


{
  "python": "3.12.13",
  "pyspark_paquete": "3.5.3",
  "motor_spark": "3.5.3",
  "pandas": "2.2.3",
  "numpy": "2.5.1",
  "matplotlib": "3.9.2",
  "faker": "40.36.0",
  "plataforma": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "sistema": "Linux",
  "java": "openjdk version \"17.0.19\" 2026-04-21",
  "fecha_utc": "2026-08-08T14:18:28+00:00"
}
Versiones del entorno (criterio 1.4): se transcriben al informe.


{'python': '3.12.13',
 'pyspark_paquete': '3.5.3',
 'motor_spark': '3.5.3',
 'pandas': '2.2.3',
 'numpy': '2.5.1',
 'matplotlib': '3.9.2',
 'faker': '40.36.0',
 'plataforma': 'Linux-6.6.122+-x86_64-with-glibc2.35',
 'sistema': 'Linux',
 'java': 'openjdk version "17.0.19" 2026-04-21',
 'fecha_utc': '2026-08-08T14:18:28+00:00'}

In [25]:
# La celda 34 (medicion.py) detiene todas las sesiones Spark al terminar, asi
# que la sesion y el crudo de la celda 22/26 ya no existen. Se recrean aqui.
from transformaciones_spark import crear_spark, leer_crudo_spark

spark = crear_spark(4, nombre="pe-u4-parquet")
raw_sp = leer_crudo_spark(spark)

import os
import time

import pandas as pd

from transformaciones_pandas import PANDAS_PATHS

# --- Parquet con pandas ---
t0 = time.perf_counter()
pd.read_csv(PANDAS_PATHS["T1"]).to_parquet("data/pandas/t1_parquet.parquet", index=False)
tp_pandas = time.perf_counter() - t0
print(f"pandas -> Parquet T1: {tp_pandas:.3f} s")

# --- Parquet con Spark ---
spark_df_t1 = TRANSFORMACIONES["T1"](raw_sp)
t0 = time.perf_counter()
spark_df_t1.write.mode("overwrite").parquet("data/spark/t1_parquet")
tp_spark = time.perf_counter() - t0
print(f"Spark  -> Parquet T1: {tp_spark:.3f} s")

def tamano_dir(d):
    return sum(os.path.getsize(os.path.join(rp, f)) for rp, _, fs in os.walk(d) for f in fs)

csv_bytes = PANDAS_PATHS["T1"].stat().st_size
parquet_bytes = tamano_dir("data/spark/t1_parquet")
print(f"CSV T1    : {csv_bytes / 1e6:.2f} MB")
print(f"Parquet   : {parquet_bytes / 1e6:.2f} MB  ({(1 - parquet_bytes / csv_bytes) * 100:.1f}% menos)")

# --- Lectura de vuelta ---
t0 = time.perf_counter()
n = spark.read.parquet("data/spark/t1_parquet").count()
print(f"Spark lee Parquet: {time.perf_counter() - t0:.3f} s ({n:,} filas)")


pandas -> Parquet T1: 1.331 s
Spark  -> Parquet T1: 7.954 s
CSV T1    : 16.45 MB
Parquet   : 6.35 MB  (61.4% menos)
Spark lee Parquet: 0.915 s (180,003 filas)


In [26]:
spark.stop()
print("Sesión Spark cerrada.")


Sesión Spark cerrada.


In [27]:
# Empaquetado final (pensado para Colab). Reutiliza las variables de las
# celdas 33 (df_equivalencia), 36 (df_amdahl), 37 (df_umbral) y 38 (versiones).
import io
import os
import re
import zipfile
from pathlib import Path

def _var(nombre):
    try:
        return globals()[nombre]
    except KeyError:
        return None

eq = _var("df_equivalencia")
amd = _var("df_amdahl")
umb = _var("df_umbral")
ver = _var("versiones")

# --- Item 4: fecha y tamano reales desde los ficheros ---
fecha = "ver data/README_dataset.md"
try:
    _md = Path("data/README_dataset.md").read_text(encoding="utf-8")
    _mf = re.search(r"fecha_generacion_utc[^`]*`[^`]*`([^`]+)`", _md)
    if _mf:
        fecha = _mf.group(1)
except Exception:
    pass
tam_tk = tickets.stat().st_size / 1e6
tam_ag = agentes.stat().st_size / 1e6

sep = "=" * 68
item = lambda t: chr(10).join([sep, t, sep])

L = []
L.append(item("TABLAS DEL INFORME PE-U4 (pandas vs PySpark, Ley de Amdahl)"))
L.append("Items 4 / 12 / 13 / 20 / 23 del CHECKLIST FINAL")
L.append("Generado por notebooks/PE_U4_pipeline_spark.ipynb (Google Colab)")
L.append("")
L.append(item("ITEM 4 [RUBRICA] - TABLA BOOKTABS DEL DATASET (criterio 1.1)"))
L.append("+------------------------------+--------------------------------------------+")
L.append("| Campo                        | Valor                                      |")
L.append("+------------------------------+--------------------------------------------+")
L.append("| Fuente                       | Sintetico (Faker es_MX + numpy/pandas, semilla fija 42) |")
L.append("| URL (generacion)             | src/generar_dataset_acc.py (Soporte-Tecnico-ACC) |")
L.append("| Licencia                     | Datos del propio equipo; codigo MIT (LICENSE) |")
L.append(f"| Fecha de generacion (UTC)    | {fecha} |")
L.append("| Registros                    | 600 000 tickets / 300 agentes |")
L.append("| Columnas                     | 15 (tickets) / 7 (agentes) |")
L.append(f"| Tamano en disco              | {tam_tk:.1f} MB / {tam_ag:.1f} MB |")
L.append("+------------------------------+--------------------------------------------+")
L.append("")
L.append(item("ITEM 12 [RUBRICA] - TABLA COMPARATIVA DE EQUIVALENCIA pandas <-> PySpark (criterio 1.4)"))
L.append(eq.to_string(index=False) if eq is not None else "PENDIENTE: ejecutar la celda 33 (modo_equivalencia)")
L.append("")
L.append(item("ITEM 13 [RUBRICA] - VERSIONES DEL ENTORNO (criterio 1.4, nivel 4)"))
L.append(chr(10).join(f"{k}: {v}" for k, v in ver.items()) if ver else "PENDIENTE: ejecutar la celda 38 (modo_versiones)")
L.append("")
L.append(item("ITEM 20 [RUBRICA] - TABLA DE AMDAHL / GUSTAFSON (criterio 2.2)"))
L.append(amd.to_string(index=False) if amd is not None else "PENDIENTE: ejecutar la celda 36 (modo_amdahl)")
L.append("")
L.append(item("ITEM 23 [RUBRICA] - TABLA DEL UMBRAL DE RENTABILIDAD (criterio 2.4)"))
L.append(umb.to_string(index=False) if umb is not None else "PENDIENTE: ejecutar la celda 37 (modo_umbral)")
L.append("Nota: la discusion razonada del umbral es texto narrativo del informe (criterio 2.4).")

Path("tablas_items_4_12_13_20_23.txt").write_text(chr(10).join(L), encoding="utf-8")
print("-> tablas_items_4_12_13_20_23.txt (completo)")

# --- Verificacion de evidencia (checklist) ---
print()
print("=== Verificacion de evidencia (checklist) ===")
artefactos = {
    "item 17: resultados/tiempos_crudos.csv": "resultados/tiempos_crudos.csv",
    "item 18: resultados/tiempos_resumen.csv": "resultados/tiempos_resumen.csv",
    "item 22: resultados/figuras/fig1_barras.png": "resultados/figuras/fig1_barras.png",
    "item 22: resultados/figuras/fig2_speedup.png": "resultados/figuras/fig2_speedup.png",
    "item 22: resultados/figuras/fig3_eficiencia.png": "resultados/figuras/fig3_eficiencia.png",
    "item 10: evidencia/spark_ui_t3.png": "evidencia/spark_ui_t3.png",
}
faltan = 0
for nombre, ruta in artefactos.items():
    ok = Path(ruta).exists()
    faltan += (not ok)
    print(("OK    " if ok else "FALTA ") + nombre)
print(f"-> {len(artefactos) - faltan}/{len(artefactos)} artefactos presentes")

# --- Empaquetado ---
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
    for carpeta in ["resultados", "evidencia"]:
        for raiz, _, archivos in os.walk(carpeta):
            for a in archivos:
                z.write(os.path.join(raiz, a))
    z.write("tablas_items_4_12_13_20_23.txt")
    z.write("data/README_dataset.md")
with open("resultados_pe_u4.zip", "wb") as f:
    f.write(buf.getvalue())
print("-> resultados_pe_u4.zip (resultados/ + evidencia/ + tablas + metadatos)")

try:
    from google.colab import files
    files.download("resultados_pe_u4.zip")
    files.download("tablas_items_4_12_13_20_23.txt")
except Exception as e:
    print("No es Colab o se cancelo la descarga:", e)


-> tablas_items_4_12_13_20_23.txt (completo)

=== Verificacion de evidencia (checklist) ===
OK    item 17: resultados/tiempos_crudos.csv
OK    item 18: resultados/tiempos_resumen.csv
FALTA item 22: resultados/figuras/fig1_barras.png
FALTA item 22: resultados/figuras/fig2_speedup.png
FALTA item 22: resultados/figuras/fig3_eficiencia.png
FALTA item 10: evidencia/spark_ui_t3.png
-> 2/6 artefactos presentes
-> resultados_pe_u4.zip (resultados/ + evidencia/ + tablas + metadatos)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 15b. Consolidacion y verificacion de salidas T1-T5 (10 CSV)

Convierte cada carpeta `data/spark/<transform>/part-*.csv` en un **CSV unico**
(`data/spark/<transform>.csv`), verifica el numero de filas esperado de los **10 CSV**
(5 de pandas + 5 de spark) y los descarga en `salidas_pandas.zip` / `salidas_spark.zip`.


In [29]:
# Consolidar T1-T5 de spark -> CSV unicos y descargarlos (1 sola vez)
import os, zipfile
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive")

root = Path("/content/drive/MyDrive/GA-SUM-05/Soporte-Tecnico-ACC")
assert (root / "data/raw/tickets.csv").exists(), f"No esta el proyecto en {root}"
os.chdir(root)

# 1) Consolidar data/spark/<carpeta> -> data/spark/<carpeta>.csv
for carpeta in ["t1_filtrado", "t2_agrupacion", "t3_join", "t4_derivada", "t5_topn"]:
    dir_f = root / "data/spark" / carpeta
    partes = sorted(p for p in dir_f.glob("part-*.csv"))
    assert partes, f"Sin partes en {dir_f}"
    destino = root / "data/spark" / (carpeta + ".csv")
    header = None
    with open(destino, "w", encoding="utf-8", newline="") as out:
        for p in partes:
            with open(p, "r", encoding="utf-8", newline="") as fh:
                h = fh.readline()
                if header is None:
                    header = h
                    out.write(h)
                elif h != header:
                    raise AssertionError(f"Header distinto en {p.name}")
                for linea in fh:
                    out.write(linea)
    n = sum(1 for _ in open(destino, encoding="utf-8")) - 1
    print(f"spark/{carpeta}.csv -> {n:,} filas")

# 2) Verificar filas esperadas
esperado = {"t1_filtrado": 180003, "t2_agrupacion": 30, "t3_join": 180003,
            "t4_derivada": 600000, "t5_topn": 10}
for carpeta, exp in esperado.items():
    n = sum(1 for _ in open(root / "data/spark" / (carpeta + ".csv"), encoding="utf-8")) - 1
    assert n == exp, f"{carpeta}: {n} != {exp}"
print("OK: filas de spark verificadas (180003/30/180003/600000/10)")

# 3) Empaquetar y descargar (2 zips, cada uno < 100 MB)
def zip_carpeta(origen, nombre_zip):
    ruta = root / nombre_zip
    with zipfile.ZipFile(ruta, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(origen.glob("t[0-9]_*.csv")):
            z.write(f, f.name)
    print(f"{nombre_zip}: {ruta.stat().st_size/1e6:.1f} MB, "
          f"{len(list(origen.glob('t[0-9]_*.csv')))} archivos")

zip_carpeta(root / "data/pandas", "salidas_pandas.zip")
zip_carpeta(root / "data/spark", "salidas_spark.zip")

files.download(str(root / "salidas_pandas.zip"))
files.download(str(root / "salidas_spark.zip"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
spark/t1_filtrado.csv -> 180,003 filas
spark/t2_agrupacion.csv -> 30 filas
spark/t3_join.csv -> 180,003 filas
spark/t4_derivada.csv -> 600,000 filas
spark/t5_topn.csv -> 10 filas
OK: filas de spark verificadas (180003/30/180003/600000/10)
salidas_pandas.zip: 14.5 MB, 5 archivos
salidas_spark.zip: 14.9 MB, 5 archivos


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>